<a href="https://colab.research.google.com/github/annasehgal/final-project-tech-68/blob/main/Anna_TECH_68_Machine_learning_for_business_in_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wildlife Strike Aircraft Damage Prediction

**Goal:** Predict whether a wildlife strike results in aircraft damage using flight characteristics, wildlife attributes, and environmental conditions. Compare multiple classification models and identify the factors most influential in predicting damage risk.

**Importance:** Wildlife strikes can create safety risks and costly aircraft damage. Identifying the factors associated with a higher risk of damage could help better understand which strike conditions are most consequential.

**Dataset:** FAA Wildlife Strike Database (`Public.xlsx`), containing reported wildlife strike incidents and information about the aircraft, flight conditions, wildlife involved, and resulting damage. The data was obtained from the FAA Wildlife Strike Database: https://wildlife.faa.gov/.

## Setup

### Standard imports

In [4]:
# data manipulation/analysis and fast multi-dimensional numerical array processing.
import pandas as pd
import numpy as np
# Plotting libraries
import seaborn as sns
import matplotlib.pyplot as plt

## Load and Understand the Data

In [14]:
import io
import zipfile
import requests

url = "https://wildlife.faa.gov/assets/database_excel.zip"

response = requests.get(url)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open("Public.xlsx") as f:
        df = pd.read_excel(f)

In [15]:
df.head()

,INDEX_NR,INCIDENT_DATE,INCIDENT_MONTH,INCIDENT_YEAR,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,LATITUDE,LONGITUDE,...,NR_INJURIES,NR_FATALITIES,COMMENTS,IMAGE,REPORTED_NAME,REPORTED_TITLE,SOURCE,PERSON,LUPDATE,TRANSFER
0,608242,1996-06-22,6,1996,NaN,NaN,KSMF,SACRAMENTO INTL,38.69542,-121.59077,...,NaN,NaN,/Legacy Record 100001/,False,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,2007-12-20,0
1,608243,1996-06-26,6,1996,NaN,NaN,KDEN,DENVER INTL AIRPORT,39.85841,-104.667,...,NaN,NaN,/Legacy Record 100002/,False,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,2007-12-20,0
2,608244,1996-07-01,7,1996,NaN,NaN,KOMA,EPPLEY AIRFIELD,41.30252,-95.89417,...,NaN,NaN,/Legacy Record 100003/,False,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,2007-12-20,0
3,608245,1996-07-01,7,1996,NaN,NaN,KIAD,WASHINGTON DULLES INTL ARPT,38.94453,-77.45581,...,NaN,NaN,/Legacy Record 100004/,False,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,2007-12-20,0
4,608246,1996-07-01,7,1996,NaN,NaN,KLGA,LA GUARDIA ARPT,40.77724,-73.87261,...,NaN,NaN,/Legacy Record 100005/,False,REDACTED,REDACTED,Air Transport Report,Air Transport Operations,2007-12-20,0


In [16]:
print("Dataset shape:", df.shape)

Dataset shape: (361180, 102)


In [17]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 361,180
Columns: 102


In [18]:
df.columns.tolist()

['INDEX_NR',
 'INCIDENT_DATE',
 'INCIDENT_MONTH',
 'INCIDENT_YEAR',
 'TIME',
 'TIME_OF_DAY',
 'AIRPORT_ID',
 'AIRPORT',
 'LATITUDE',
 'LONGITUDE',
 'RUNWAY',
 'STATE',
 'FAAREGION',
 'LOCATION',
 'OPID',
 'OPERATOR',
 'REG',
 'FLT',
 'AIRCRAFT',
 'AMA',
 'AMO',
 'EMA',
 'EMO',
 'AC_CLASS',
 'AC_MASS',
 'TYPE_ENG',
 'NUM_ENGS',
 'ENG_1_POS',
 'ENG_2_POS',
 'ENG_3_POS',
 'ENG_4_POS',
 'PHASE_OF_FLIGHT',
 'HEIGHT',
 'SPEED',
 'DISTANCE',
 'SKY',
 'PRECIPITATION',
 'AOS',
 'COST_REPAIRS',
 'COST_OTHER',
 'COST_REPAIRS_INFL_ADJ',
 'COST_OTHER_INFL_ADJ',
 'INGESTED_OTHER',
 'INDICATED_DAMAGE',
 'DAMAGE_LEVEL',
 'STR_RAD',
 'DAM_RAD',
 'STR_WINDSHLD',
 'DAM_WINDSHLD',
 'STR_NOSE',
 'DAM_NOSE',
 'STR_ENG1',
 'DAM_ENG1',
 'ING_ENG1',
 'STR_ENG2',
 'DAM_ENG2',
 'ING_ENG2',
 'STR_ENG3',
 'DAM_ENG3',
 'ING_ENG3',
 'STR_ENG4',
 'DAM_ENG4',
 'ING_ENG4',
 'STR_PROP',
 'DAM_PROP',
 'STR_WING_ROT',
 'DAM_WING_ROT',
 'STR_FUSE',
 'DAM_FUSE',
 'STR_LG',
 'DAM_LG',
 'STR_TAIL',
 'DAM_TAIL',
 'STR_LGHTS',


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 361180 entries, 0 to 361179
Columns: 102 entries, INDEX_NR to TRANSFER
dtypes: bool(1), float64(17), int64(41), object(43)
memory usage: 278.7+ MB


In [21]:
column_summary = pd.DataFrame({"dtype": df.dtypes, "non_null": df.notna().sum(), "missing": df.isna().sum(), "missing_pct": (df.isna().mean() * 100).round(1), "unique": df.nunique()})
column_summary

,dtype,non_null,missing,missing_pct,unique
INDEX_NR,int64,361180,0,0.0,361172
INCIDENT_DATE,object,361180,0,0.0,13374
INCIDENT_MONTH,int64,361180,0,0.0,12
INCIDENT_YEAR,int64,361180,0,0.0,37
TIME,object,239890,121290,33.6,1441
...,...,...,...,...,...
REPORTED_TITLE,object,361180,0,0.0,1
SOURCE,object,361180,0,0.0,16
PERSON,object,340093,21087,5.8,6
LUPDATE,object,361180,0,0.0,4929


In [22]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
INDEX_NR,361180.0,1.079791e+06,4.237904e+05,608242.00,711208.75,819188.5,1506732.25,1940993.0
INCIDENT_MONTH,361180.0,7.176541e+00,2.744886e+00,1.00,5.00,7.0,9.00,12.0
INCIDENT_YEAR,361180.0,2.014303e+03,9.175313e+00,1990.00,2008.00,2016.0,2022.00,2026.0
EMO,226289.0,8.038327e+00,1.141427e+01,1.00,1.00,4.0,10.00,97.0
AC_MASS,258644.0,3.566663e+00,8.623508e-01,1.00,3.00,4.0,4.00,5.0
NUM_ENGS,258345.0,2.000890e+00,4.031844e-01,1.00,2.00,2.0,2.00,4.0
ENG_1_POS,258320.0,2.680381e+00,2.113312e+00,1.00,1.00,1.0,5.00,7.0
ENG_2_POS,240992.0,2.495900e+00,1.945138e+00,1.00,1.00,1.0,5.00,7.0
ENG_3_POS,13743.0,2.901332e+00,1.937162e+00,1.00,1.00,1.0,5.00,6.0
ENG_4_POS,3685.0,2.160923e+00,1.466748e+00,1.00,1.00,1.0,4.00,5.0


In [23]:
# since there are 102 columns, I want to understand values under each column
for col in df.columns:
    print(f"\n--- {col} ({df[col].nunique()} unique) ---")
    print(df[col].value_counts(dropna=False).head(20))


--- INDEX_NR (361172 unique) ---
INDEX_NR
793166     2
1850863    2
1520761    2
1859904    2
814615     2
1850036    2
778860     2
1861841    2
1345173    1
1345172    1
1345171    1
1345170    1
1345169    1
1345167    1
1345166    1
1345174    1
1345164    1
1345163    1
1345162    1
1345161    1
Name: count, dtype: int64

--- INCIDENT_DATE (13374 unique) ---
INCIDENT_DATE
2025-10-08    197
2025-09-26    195
2025-09-25    173
2025-10-09    169
2024-10-24    157
2025-09-28    154
2026-05-04    152
2026-09-04    151
2026-07-29    151
2024-10-07    150
2025-09-07    148
2025-09-08    148
2026-09-05    147
2025-09-27    145
2023-10-05    144
2026-08-27    143
2024-10-01    143
2024-09-20    142
2026-08-17    142
2024-09-19    141
Name: count, dtype: int64

--- INCIDENT_MONTH (12 unique) ---
INCIDENT_MONTH
8     51633
9     47520
7     47143
10    43800
5     36742
6     30952
4     26387
11    23840
3     17257
12    13394
1     11455
2     11057
Name: count, dtype: int64

--- INCIDEN

In [25]:
df.describe()

,INDEX_NR,INCIDENT_MONTH,INCIDENT_YEAR,EMO,AC_MASS,NUM_ENGS,ENG_1_POS,ENG_2_POS,ENG_3_POS,ENG_4_POS,...,STR_LGHTS,DAM_LGHTS,STR_OTHER,DAM_OTHER,OUT_OF_RANGE_SPECIES,REMAINS_COLLECTED,REMAINS_SENT,NR_INJURIES,NR_FATALITIES,TRANSFER
count,3.611800e+05,361180.000000,361180.000000,226289.000000,258644.000000,258345.000000,258320.000000,240992.000000,13743.000000,3685.000000,...,361180.000000,361180.000000,361180.000000,361180.000000,361180.000000,361180.000000,361180.000000,308.000000,25.000000,361180.0
mean,1.079791e+06,7.176541,2014.303178,8.038327,3.566663,2.000890,2.680381,2.495900,2.901332,2.160923,...,0.004552,0.002999,0.140431,0.007221,0.001094,0.544114,0.185977,1.279221,2.080000,0.0
std,4.237904e+05,2.744886,9.175313,11.414271,0.862351,0.403184,2.113312,1.945138,1.937162,1.466748,...,0.067313,0.054677,0.347434,0.084668,0.033052,0.498051,0.389088,0.660636,1.630951,0.0
min,6.082420e+05,1.000000,1990.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.0
25%,7.112088e+05,5.000000,2008.000000,1.000000,3.000000,2.000000,1.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.0
50%,8.191885e+05,7.000000,2016.000000,4.000000,4.000000,2.000000,1.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,2.000000,0.0
75%,1.506732e+06,9.000000,2022.000000,10.000000,4.000000,2.000000,5.000000,5.000000,5.000000,4.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,2.000000,0.0
max,1.940993e+06,12.000000,2026.000000,97.000000,5.000000,4.000000,7.000000,7.000000,6.000000,5.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,7.000000,8.000000,0.0


In [27]:
# see all values
for col in df.columns:
    print(f"\n--- {col} ({df[col].nunique()} unique) ---")
    print(df[col].value_counts(dropna=False))


--- INDEX_NR (361172 unique) ---
INDEX_NR
793166     2
1850863    2
1520761    2
1859904    2
814615     2
          ..
743787     1
743786     1
743785     1
743784     1
743797     1
Name: count, Length: 361172, dtype: int64

--- INCIDENT_DATE (13374 unique) ---
INCIDENT_DATE
2025-10-08    197
2025-09-26    195
2025-09-25    173
2025-10-09    169
2024-10-24    157
             ... 
2026-10-01      1
2026-09-22      1
2026-09-19      1
2026-09-30      1
2026-10-03      1
Name: count, Length: 13374, dtype: int64

--- INCIDENT_MONTH (12 unique) ---
INCIDENT_MONTH
8     51633
9     47520
7     47143
10    43800
5     36742
6     30952
4     26387
11    23840
3     17257
12    13394
1     11455
2     11057
Name: count, dtype: int64

--- INCIDENT_YEAR (37 unique) ---
INCIDENT_YEAR
2025    24457
2024    22370
2023    19623
2026    17608
2019    17345
2022    17223
2018    16206
2021    15642
2017    14754
2015    13777
2014    13690
2016    13380
2020    11627
2013    11405
2012    10937
2

# Pre-process, cleaning and EDA

`INDEX_NR` has 361,172 unique values across 361,180 rows, so we check for repeated IDs.

In [28]:
df[df["INDEX_NR"].duplicated(keep=False)].sort_values("INDEX_NR")

,INDEX_NR,INCIDENT_DATE,INCIDENT_MONTH,INCIDENT_YEAR,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,LATITUDE,LONGITUDE,...,NR_INJURIES,NR_FATALITIES,COMMENTS,IMAGE,REPORTED_NAME,REPORTED_TITLE,SOURCE,PERSON,LUPDATE,TRANSFER
152290,778860,2016-03-01,3,2016,NaN,NaN,KPSM,PORTSMOUTH INTL ARPT AT PEASE,43.07796,-70.82327,...,NaN,NaN,2016-3-1-211917 /Legacy Record 371717/,False,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,2016-09-01,0
152291,778860,2016-03-01,3,2016,NaN,NaN,KPSM,PORTSMOUTH INTL ARPT AT PEASE,43.07796,-70.82327,...,NaN,NaN,2016-3-1-211917 /Legacy Record 371717/,False,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,2016-09-01,0
162193,793166,2017-04-29,4,2017,NaN,NaN,KNUQ,MOFFETT FEDERAL AIRFIELD,37.41614,-122.04914,...,NaN,NaN,2017-6-12-173440 /Legacy Record 390497/,False,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,2018-05-18,0
162194,793166,2017-04-29,4,2017,NaN,NaN,KNUQ,MOFFETT FEDERAL AIRFIELD,37.41614,-122.04914,...,NaN,NaN,2017-6-12-173440 /Legacy Record 390497/,False,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,2018-05-18,0
177224,814615,2018-09-27,9,2018,NaN,NaN,KBDL,BRADLEY INTL,41.93887,-72.68323,...,NaN,NaN,2018-9-28-111208 /Legacy Record 416060/,False,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,2018-11-27,0
177225,814615,2018-09-27,9,2018,NaN,NaN,KBDL,BRADLEY INTL,41.93887,-72.68323,...,NaN,NaN,2018-9-28-111208 /Legacy Record 416060/,False,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,2018-11-27,0
277917,1520761,2017-03-04,3,2017,NaN,NaN,KISN,SLOULIN FLD INTL ARPT,48.17794,-103.64235,...,NaN,NaN,NaN,False,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,2024-05-15,0
277918,1520761,2017-03-04,3,2017,NaN,NaN,KISN,SLOULIN FLD INTL ARPT,48.17794,-103.64235,...,NaN,NaN,NaN,False,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,2024-05-15,0
343496,1850036,2026-02-13,2,2026,,NaN,KPAO,PALO ALTO ARPT OF SANTA CLARA COUNTY,37.46112,-122.11504,...,NaN,NaN,revised species,False,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,2026-05-29,0
343497,1850036,2026-02-13,2,2026,,NaN,KPAO,PALO ALTO ARPT OF SANTA CLARA COUNTY,37.46112,-122.11504,...,NaN,NaN,revised species,False,REDACTED,REDACTED,FAA Form 5200-7-E,Carcass Found,2026-05-29,0


In [29]:
df.duplicated().sum()

np.int64(0)

Since there are no duplicated `INDEX_NR` there are no identical rows in the dataset.

In [30]:
df["INDICATED_DAMAGE"].value_counts()

,count
INDICATED_DAMAGE,
0,338813
1,22367
